# Eksplorasi Data & Analisis (EDA) Dataset Riil Gaji Profesional Data & AI
**Kelompok:** GodBless (Revaldo Fransisco Hohary, Samuel Natanael, Indira Mai Narwastu)  
**Mata Kuliah:** IT6043 – Machine Learning (FTI UKDW)  
**Tujuan Proyek:** Prediksi Gaji Tahunan (`salary_in_usd`) Profesional Bidang Data, AI, dan Teknologi Menggunakan Model Regresi Machine Learning.

---
### Ringkasan Alur Analisis
1. **Memuat Dataset Riil (`ai_jobs_salaries_clean.csv`)**: Inspeksi dimensi, struktur 17 kolom, dan sampel data.
2. **Validasi Kualitas Data & Kolom Tambahan**:
   - Pengecekan status duplikasi (sudah bersih, 0 duplikat).
   - Analisis *missing value* (hanya ditemukan pada kolom `isco_group_hint` sebanyak 40.540 baris).
   - Penjelasan fitur hasil *enrichment*: `role_family`, `work_mode`, `salary_outlier_flag`, `experience_level_label`, `employment_type_label`.
   - Pencegahan *Data Leakage* (`salary`, `salary_currency`, dan `salary_outlier_flag`).
3. **Analisis Statistik Deskriptif Target** (`salary_in_usd`).
4. **Visualisasi Distribusi & Analisis Outlier (Histogram & Boxplot)**.
5. **Eksplorasi Fitur Kategorikal & Klasterisasi Peran (`role_family`)**:
   - Distribusi tingkat pengalaman, ukuran organisasi, tahun, dan sistem kerja.
   - Analisis klaster peran (`role_family`) untuk mengatasi masalah kardinalitas tinggi `job_title`.
6. **Analisis Bivariat & Heatmap Korelasi**:
   - Median kompensasi per level pengalaman, ukuran perusahaan, dan peran.
   - Heatmap korelasi matriks numerik & ordinal.
7. **Rangkuman Pipeline Pre-processing & Rencana Model ML**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi visualisasi
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Library visualisasi & analisis data berhasil diimpor.")


## 1. Memuat Dataset Riil (`ai_jobs_salaries_clean.csv`)
Dataset yang dimuat merupakan data bersih hasil kurasi sebanyak 71.913 baris observasi unik dan 17 kolom.


In [ ]:
# Membaca dataset riil
df = pd.read_csv('ai_jobs_salaries_clean.csv')

print(f"Dimensi dataset riil: {df.shape[0]:,} baris dan {df.shape[1]} kolom\n")
df.info()


In [ ]:
# Menampilkan 5 baris pertama data riil
df.head()


## 2. Pemeriksaan Kualitas Data & Validasi Fitur
Melakukan pengecekan terhadap nilai kosong (*missing value*), duplikat, serta mengidentifikasi fitur-fitur baru hasil pengayaan (*feature enrichment*).


In [ ]:
# 2.1 Cek Missing Values
missing_series = df.isnull().sum()
print("Jumlah Missing Values per Kolom:")
print(missing_series[missing_series > 0])
if missing_series.sum() == 0:
    print("Tidak ditemukan missing value pada seluruh kolom.")
else:
    print(f"\nCatatan: Kolom 'isco_group_hint' memiliki {missing_series['isco_group_hint']:,} missing values ({missing_series['isco_group_hint']/len(df)*100:.2f}%).")


In [ ]:
# 2.2 Verifikasi Duplikasi Data
print(f"Jumlah baris duplikat: {df.duplicated().sum()} (Dataset sudah berstatus deduped/bersih)")


### Catatan Validasi Data & Penanganan Kebocoran Informasi (*Data Leakage*):
1. **Fitur Tambahan yang Sangat Bermanfaat:**
   - `role_family`: Mengelompokkan 422 `job_title` yang tersebar menjadi kategori peran yang lebih terstruktur (`Data Analyst`, `Data Engineer`, `Data Scientist`, `ML Engineer`, dll.). Ini mempermudah pemodelan dan *encoding*.
   - `work_mode`: Label deskriptif dari `remote_ratio` (`On-site`, `Remote`, `Hybrid`).
   - `experience_level_label` & `employment_type_label`: Versi label lengkap untuk mempermudah analisis dan visualisasi.
2. **Pencegahan Data Leakage (Wajib di-drop saat modeling):**
   - `salary` & `salary_currency`: Hubungan matematis deterministik langsung ke `salary_in_usd`.
   - `salary_outlier_flag`: Dibuat berdasarkan nilai target `salary_in_usd`, sehingga **wajib dibuang** sebelum pelatihan model agar tidak terjadi *target leakage*.
   - `isco_group_hint`: Memiliki missing value sebanyak 56,37% dan bersifat redundant dengan `role_family`, sehingga fitur ini sebaiknya di-drop.


## 3. Statistik Deskriptif Variabel Target (`salary_in_usd`)
Analisis parameter statistik dari target kontinu yang akan diprediksi oleh model regresi.


In [ ]:
target_stats = df['salary_in_usd'].describe()
skew_val = df['salary_in_usd'].skew()

print("=== STATISTIK DESKRIPTIF TARGET (salary_in_usd) ===")
print(f"Jumlah Observasi : {int(target_stats['count']):,}")
print(f"Rata-rata (Mean) : ${target_stats['mean']:,.2f}")
print(f"Median (Q2)      : ${target_stats['50%']:,.2f}")
print(f"Standar Deviasi  : ${target_stats['std']:,.2f}")
print(f"Kuartil 1 (Q1)   : ${target_stats['25%']:,.2f}")
print(f"Kuartil 3 (Q3)   : ${target_stats['75%']:,.2f}")
print(f"Nilai Minimum    : ${target_stats['min']:,.2f}")
print(f"Nilai Maksimum   : ${target_stats['max']:,.2f}")
print(f"Skewness         : {skew_val:.4f} (Right-Skewed / Condong ke Kanan)")


## 4. Visualisasi Distribusi Target & Deteksi Outlier (Histogram & Boxplot)
Menampilkan visualisasi sebaran frekuensi dan sebaran outlier pada variabel `salary_in_usd`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram salary_in_usd
sns.histplot(df['salary_in_usd'], kde=True, bins=40, ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribusi Frekuensi salary_in_usd (Histogram & KDE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('salary_in_usd (USD)')
axes[0].set_ylabel('Frekuensi')

# Boxplot salary_in_usd
sns.boxplot(x=df['salary_in_usd'], ax=axes[1], color='#5B84B1')
axes[1].set_title('Deteksi Outlier salary_in_usd (Boxplot)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('salary_in_usd (USD)')

plt.tight_layout()
plt.show()


In [ ]:
# Verifikasi Outlier dengan kolom 'salary_outlier_flag'
outlier_counts = df['salary_outlier_flag'].value_counts()
print("=== DETEKSI OUTLIER METODE IQR ===")
print(f"Batas Atas Outlier (IQR)  : $331,787.50")
print(f"Jumlah Data Non-Outlier   : {outlier_counts[False]:,} baris ({outlier_counts[False]/len(df)*100:.2f}%)")
print(f"Jumlah Data Outlier       : {outlier_counts[True]:,} baris ({outlier_counts[True]/len(df)*100:.2f}%)")


### Insight Visualisasi Distribusi:
- **Histogram:** Data memiliki kemiringan positif (*right-skewed*) dengan skewness 1,48. Sebagian besar gaji berada pada rentang 80.000–180.000 USD dengan ekor panjang ke kanan hingga 800.000 USD. Solusi pre-processing terbaik adalah menerapkan transformasi logaritmik $\log(1 + y)$ (`np.log1p`).
- **Boxplot:** Terdapat 1.754 data outlier di atas batas \$331.787. Outlier ini merupakan kompensasi riil posisi eksekutif (*C-level/Director*) atau staf ahli senior di perusahaan besar tier-1, bukan kesalahan input data.


## 5. Eksplorasi Fitur Kategorikal & Klasterisasi Peran (`role_family`)
Menganalisis sebaran kategori pada variabel utama dan kelompok peran kerja.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Experience Level
sns.countplot(data=df, x='experience_level_label', order=['Entry-level', 'Mid-level', 'Senior-level', 'Executive-level'], ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('Distribusi Tingkat Pengalaman', fontweight='bold')
axes[0, 0].set_xlabel('Tingkat Pengalaman')
axes[0, 0].set_ylabel('Jumlah')

# Work Year
sns.countplot(data=df, x='work_year', ax=axes[0, 1], palette='viridis')
axes[0, 1].set_title('Distribusi Tahun Pencatatan (work_year)', fontweight='bold')
axes[0, 1].set_xlabel('Tahun')
axes[0, 1].set_ylabel('Jumlah')

# Company Size
sns.countplot(data=df, x='company_size', order=['S', 'M', 'L'], ax=axes[1, 0], palette='Greens_r')
axes[1, 0].set_title('Distribusi Ukuran Perusahaan (company_size)', fontweight='bold')
axes[1, 0].set_xlabel('Ukuran Perusahaan (S: Kecil, M: Menengah, L: Besar)')
axes[1, 0].set_ylabel('Jumlah')

# Work Mode
sns.countplot(data=df, x='work_mode', order=['On-site', 'Remote', 'Hybrid'], ax=axes[1, 1], palette='mako')
axes[1, 1].set_title('Distribusi Sistem Kerja (work_mode)', fontweight='bold')
axes[1, 1].set_xlabel('Mode Kerja')
axes[1, 1].set_ylabel('Jumlah')

plt.tight_layout()
plt.show()


In [ ]:
# Distribusi Klaster Peran (role_family)
plt.figure(figsize=(10, 5))
role_counts = df['role_family'].value_counts()
sns.barplot(x=role_counts.values, y=role_counts.index, palette='rocket')
plt.title('Distribusi Klaster Peran Pekerjaan (role_family)', fontweight='bold')
plt.xlabel('Jumlah Observasi')
plt.ylabel('Klaster Peran')
plt.show()


## 6. Analisis Bivariat & Heatmap Korelasi
Menganalisis bagaimana variabel fitur memengaruhi nilai median gaji tahunan.


In [ ]:
# 6.1 Median Gaji per Tingkat Pengalaman
exp_salary = df.groupby('experience_level')['salary_in_usd'].median().reindex(['EN', 'MI', 'SE', 'EX'])
print("Median Gaji per Tingkat Pengalaman:")
for level, val in exp_salary.items():
    print(f"- {level:2s} : ${val:,.2f}")

# 6.2 Median Gaji Perusahaan US vs Non-US
is_us = (df['company_location'] == 'US')
print(f"\nMedian Gaji Perusahaan US     : ${df[is_us]['salary_in_usd'].median():,.2f}")
print(f"Median Gaji Perusahaan Non-US : ${df[~is_us]['salary_in_usd'].median():,.2f}")

# 6.3 Median Gaji per Klaster Peran
print(f"\nMedian Gaji per Klaster Peran (role_family):")
print(df.groupby('role_family')['salary_in_usd'].median().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))


In [ ]:
# Heatmap Korelasi Fitur Numerik & Ordinal
df_corr = df[['work_year', 'remote_ratio', 'salary_in_usd']].copy()
df_corr['exp'] = df['experience_level'].map({'EN': 1, 'MI': 2, 'SE': 3, 'EX': 4})
df_corr['size'] = df['company_size'].map({'S': 1, 'M': 2, 'L': 3})

corr_matrix = df_corr.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', vmin=-0.2, vmax=1.0, cbar=True)
plt.title('Heatmap Korelasi Antar Fitur terhadap salary_in_usd', fontsize=12, fontweight='bold')
plt.show()


### Insight Analisis Bivariat & Heatmap:
1. **Fitur Pengalaman (`exp` / `experience_level`):** Korelasi linear positif terkuat dengan gaji ($r = 0,320$). Kenaikan median konsisten dari Entry (\$85.469) $\rightarrow$ Mid (\$122.000) $\rightarrow$ Senior (\$156.160) $\rightarrow$ Executive (\$190.000).
2. **Lokasi Perusahaan (`company_location`):** Perusahaan AS menawarkan gaji median (\$147.200) yang jauh lebih tinggi dibanding perusahaan non-AS (\$90.000). Ini memvalidasi pembuatan fitur binary `is_us_company`.
3. **Klaster Peran (`role_family`):** Memberikan pemisahan gaji yang jelas antar domain pekerjaan (misal: ML Engineer & Data Scientist memiliki median lebih tinggi daripada Data Analyst umum).


## 7. Rangkuman Pipeline Pre-processing & Rencana Model ML
Berdasarkan data riil yang telah dianalisis, pipeline pemrosesan data untuk model machine learning adalah sebagai berikut:

| Karakteristik / Masalah Data | Temuan pada Data Riil | Solusi Pre-processing yang Diterapkan |
| :--- | :--- | :--- |
| **Status Duplikasi** | 0 duplikat (sudah bersih 71.913 baris) | Langsung siap diproses ke partisi train/test split. |
| **Pencegahan Data Leakage** | Kolom `salary`, `salary_currency`, & `salary_outlier_flag` | **Wajib di-drop** sebelum pelatihan model. |
| **Fitur Missing Tinggi** | `isco_group_hint` (56,37% missing) | Dibuang (*drop*) karena redundant dengan `role_family`. |
| **Skewness Target** | `salary_in_usd` condong ke kanan (skew = 1,48) | Transformasi logaritmik $\log(1 + y)$ (`np.log1p`) pada data latih. |
| **Kardinalitas Kategori** | Fitur baru `role_family` tersedia | Menggunakan `role_family` + *One-Hot Encoding* untuk efisiensi komputasi. |
| **Encoding Ordinal** | `experience_level` dan `company_size` | *Ordinal Encoding* (`EN<MI<SE<EX` & `S<M<L`). |
| **Model ML yang Dibandingkan** | Tugas regresi tabular | **Ridge Regression**, **Random Forest Regressor**, dan **XGBoost Regressor**. |
| **Program Client (GUI)** | Kebutuhan antarmuka pengguna | Model terbaik diekspor ke format `.joblib` dan dihubungkan ke aplikasi interaktif berbasis **Streamlit**. |
